In [1]:
library('dplyr')

Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"

Attachement du package : 'dplyr'


Les objets suivants sont masqués depuis 'package:stats':

    filter, lag


Les objets suivants sont masqués depuis 'package:base':

    intersect, setdiff, setequal, union




There are 38 basic relevant variables. For the calculation of the entropy, we will use our linear model. Let's take a look at our 38 base variables and what is left after applying our pre-processing and feature selection strategy for the linear model.

In [2]:
train_values <- read.csv("train_values.csv",stringsAsFactors = T)
test_values <- read.csv("test_values.csv",stringsAsFactors = T)
train_labels <- read.csv("train_labels.csv",stringsAsFactors = T)

In [2]:
# The 38 relevant features are as follows
feature_initial <- colnames(test_values[,-1])
feature_initial

[1] "geo_level_1_id"                        
 [2] "geo_level_2_id"                        
 [3] "geo_level_3_id"                        
 [4] "count_floors_pre_eq"                   
 [5] "age"                                   
 [6] "area_percentage"                       
 [7] "height_percentage"                     
 [8] "land_surface_condition"                
 [9] "foundation_type"                       
[10] "roof_type"                             
[11] "ground_floor_type"                     
[12] "other_floor_type"                      
[13] "position"                              
[14] "plan_configuration"                    
[15] "has_superstructure_adobe_mud"          
[16] "has_superstructure_mud_mortar_stone"   
[17] "has_superstructure_stone_flag"         
[18] "has_superstructure_cement_mortar_stone"
[19] "has_superstructure_mud_mortar_brick"   
[20] "has_superstructure_cement_mortar_brick"
[21] "has_superstructure_timber"             
[22] "has_superstructure_bamboo"             
[23] "has_superstructure_rc_non_engineered"  
[24] "has_superstructure_rc_engineered"      
[25] "has_superstructure_other"              
[26] "legal_ownership_status"                
[27] "count_families"                        
[28] "has_secondary_use"                     
[29] "has_secondary_use_agriculture"         
[30] "has_secondary_use_hotel"               
[31] "has_secondary_use_rental"              
[32] "has_secondary_use_institution"         
[33] "has_secondary_use_school"              
[34] "has_secondary_use_industry"            
[35] "has_secondary_use_health_post"         
[36] "has_secondary_use_gov_office"          
[37] "has_secondary_use_use_police"          
[38] "has_secondary_use_other"

In [3]:
length(feature_initial)

[1] 38

In [2]:
# We apply our pre-processing and feature selection pipline for our linear model# 
dataNN <- read.csv("data_target_encoding_NN.csv",stringsAsFactors = T)
testNN <- read.csv("test_target_encoding_NN.csv",stringsAsFactors = T)
dataNN.lin <- dataNN[,-c(34,37,42,45,50,54,58,68)]
testNN.lin <- testNN[,-c(34,37,42,45,50,54,58,68)]
dataNN.lin <- dataNN.lin[,-c(13,14,19,20,21,24,25,26,27,28,29,30,31,32,35,44,46,51,52,53,54,55,56,57,58,59,60,61,62)]
testNN.lin <- testNN.lin[,-c(13,14,19,20,21,24,25,26,27,28,29,30,31,32,35,44,46,51,52,53,54,55,56,57,58,59,60,61,62)]
dataNN.lin <- dataNN.lin[,-c(12,24,28,15,31,10,32)]
testNN.lin <- testNN.lin[,-c(12,24,28,15,31,10,32)]

In [19]:
# Along the way, we replaced some variables with others, via target encoding or dummy encoding and deleted some of its variables. 
# At the end of this process the remaining variables are as follows :
feature_final <- colnames(testNN.lin)
feature_final

[1] "geo_level_1_mean_damage"               
 [2] "geo_level_1_sd_damage"                 
 [3] "geo_level_2_mean_damage"               
 [4] "geo_level_2_sd_damage"                 
 [5] "geo_level_3_mean_damage"               
 [6] "geo_level_3_sd_damage"                 
 [7] "count_floors_pre_eq"                   
 [8] "age"                                   
 [9] "area_percentage"                       
[10] "has_superstructure_adobe_mud"          
[11] "has_superstructure_mud_mortar_brick"   
[12] "has_superstructure_cement_mortar_brick"
[13] "has_superstructure_bamboo"             
[14] "count_families"                        
[15] "has_secondary_use_agriculture"         
[16] "land_surface_condition_n"              
[17] "land_surface_condition_t"              
[18] "foundation_type_r"                     
[19] "foundation_type_u"                     
[20] "foundation_type_w"                     
[21] "roof_type_x"                           
[22] "ground_floor_type_f"                   
[23] "ground_floor_type_v"                   
[24] "other_floor_type_j"                    
[25] "other_floor_type_x"                    
[26] "position_t"

In [20]:
length(feature_final)

[1] 26

We will now give a ranking of the importance of the remaining variables via $H(y|X^{-i}) - H(y|X) = 0.$ Where, H(y|X) denotes the entropy of our predictor to which we have made the feature X eat. 

We start by computing the entropy of $H(y|X)$. To do this we start by training our model on the 18 features, as done previously. To make a prediction on the testing set. 

In [3]:
X.train <- as.matrix(dataNN.lin[,setdiff(colnames(dataNN.lin),c("damage_grade_X1","damage_grade_X2","damage_grade_X3"))])
X.test <- as.matrix(testNN.lin)
y.train <- as.matrix(dataNN.lin[,c("damage_grade_X1","damage_grade_X2","damage_grade_X3")])

X.train <- cbind(matrix(1, nr = nrow(X.train), nc = 1),X.train)
beta_hat <- solve(t(X.train)%*%X.train)%*%t(X.train)%*%y.train
y.test <- cbind(matrix(1, nr = nrow(X.test), nc = 1),X.test) %*% beta_hat
pred.lin <- 1:nrow(y.test)
for (i in pred.lin){ pred.lin[i] <- which.max(y.test[i,])}

In [6]:
df <- as.data.frame(pred.lin)
colnames(df) <- c('damage_grade')

df <- df %>% group_by(damage_grade) %>% 
  summarise('H' =n (),
            .groups = 'drop')

for (i in 1:3){df$H[i] <- df$H[i]/86868}
df

damage_grade,H
<int>,<dbl>
1,0.02957361
2,0.63616061
3,0.33426578


We calculate the entropy via the formula : $$H(y|X) = - \sum_y p(y) log(p(y)) $$

In [7]:
H <- - df$H[1]*log(df$H[1]) - df$H[2]*log(df$H[2]) - df$H[3]*log(df$H[3])
H

[1] 0.7581578

We now proceed to the calculation of entropy $H(y|X^{-i})$.

In [8]:
for (i in 1:26){
    dataNN.lin.copy <- dataNN.lin[,-i]
    testNN.lin.copy <- testNN.lin[,-i]
   
    X.train <- as.matrix(dataNN.lin.copy[,setdiff(colnames(dataNN.lin.copy),c("damage_grade_X1","damage_grade_X2","damage_grade_X3"))])
    X.test <- as.matrix(testNN.lin.copy)
    y.train <- as.matrix(dataNN.lin.copy[,c("damage_grade_X1","damage_grade_X2","damage_grade_X3")])

    X.train <- cbind(matrix(1, nr = nrow(X.train), nc = 1),X.train)
    beta_hat <- solve(t(X.train)%*%X.train)%*%t(X.train)%*%y.train
    y.test <- cbind(matrix(1, nr = nrow(X.test), nc = 1),X.test) %*% beta_hat
    pred.lin.copy <- 1:nrow(y.test)
    for (j in pred.lin.copy){ pred.lin.copy[j] <- which.max(y.test[j,])}
    
    ddf <- as.data.frame(pred.lin.copy)
    colnames(ddf) <- c('damage_grade')

    ddf <- ddf %>% group_by(damage_grade) %>% 
    summarise( 'A' = n(),
                .groups = 'drop')

    for (k in 1:3){ddf$A[k] <- ddf$A[k]/86868}

    df <- merge(df,ddf,by=c('damage_grade','damage_grade'),all.x=T)
    names(df)[i+2] = paste("H", i, sep = "")
}
df

damage_grade,H,H1,H2,H3,H4,H5,H6,H7,H8,⋯,H17,H18,H19,H20,H21,H22,H23,H24,H25,H26
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.02957361,0.02953907,0.02950454,0.02949302,0.02993047,0.03033338,0.0299650,0.0296657,0.03073629,⋯,0.02953907,0.02226366,0.02283925,0.02635032,0.02869872,0.02957361,0.0287793,0.02895197,0.02964268,0.02963116
2,0.63616061,0.63757655,0.63631026,0.63789888,0.64069623,0.64554266,0.6521964,0.6362412,0.63499793,⋯,0.63620666,0.64168624,0.64164019,0.63870470,0.63673620,0.63627573,0.6364714,0.63719667,0.63636782,0.63614910
3,0.33426578,0.33288438,0.33418520,0.33260810,0.32937330,0.32412396,0.3178386,0.3340931,0.33426578,⋯,0.33425427,0.33605010,0.33552056,0.33494497,0.33456509,0.33415067,0.3347493,0.33385136,0.33398950,0.33421974


In [19]:
entropie <- 0:26
for (i in entropie){ entropie[i+1] <-  -df[1,2+i]*log(df[1,2+i]) -df[2,2+i]*log(df[2,2+i]) -df[3,2+i]*log(df[3,2+i])}
entropie <- entropie - entropie[1] 
entropie <- entropie[2:27]
M <- matrix(0,26,2)
M[,1] <- entropie
M[,2] <- 1:26
M <- as.data.frame(M)
colnames(M) <- c('entropie','indice')
M <- M[order(M$entropie, decreasing = TRUE),]
M

,entropie,indice
,<dbl>,<dbl>
8,3.544150e-03,8
7,1.712837e-04,7
26,1.469314e-04,26
10,1.327375e-04,10
25,3.392743e-05,25
22,-7.410984e-05,22
17,-1.134035e-04,17
13,-2.150852e-04,13
16,-2.468669e-04,16


In [17]:
entropie

[1] -9.993818e-04 -2.639108e-04 -1.320617e-03 -2.107579e-03 -4.428952e-03
 [6] -9.983567e-03  1.712837e-04  3.544150e-03 -3.380754e-04  1.327375e-04
[11] -2.726853e-03 -5.897041e-04 -2.150852e-04 -2.884761e-04 -4.771800e-04
[16] -2.468669e-04 -1.134035e-04 -2.230021e-02 -2.071590e-02 -9.641970e-03
[21] -2.505520e-03 -7.410984e-05 -2.137446e-03 -2.181889e-03  3.392743e-05
[26]  1.469314e-04

In [26]:
colnames(dataNN.lin)[M$indice]

[1] "age"                                   
 [2] "count_floors_pre_eq"                   
 [3] "position_t"                            
 [4] "has_superstructure_adobe_mud"          
 [5] "other_floor_type_x"                    
 [6] "ground_floor_type_f"                   
 [7] "land_surface_condition_t"              
 [8] "has_superstructure_bamboo"             
 [9] "land_surface_condition_n"              
[10] "geo_level_1_sd_damage"                 
[11] "count_families"                        
[12] "area_percentage"                       
[13] "has_secondary_use_agriculture"         
[14] "has_superstructure_cement_mortar_brick"
[15] "geo_level_1_mean_damage"               
[16] "geo_level_2_mean_damage"               
[17] "geo_level_2_sd_damage"                 
[18] "ground_floor_type_v"                   
[19] "other_floor_type_j"                    
[20] "roof_type_x"                           
[21] "has_superstructure_mud_mortar_brick"   
[22] "geo_level_3_mean_damage"               
[23] "foundation_type_w"                     
[24] "geo_level_3_sd_damage"                 
[25] "foundation_type_u"                     
[26] "foundation_type_r"

In [27]:
M$indice

[1]  8  7 26 10 25 22 17 13 16  2 14  9 15 12  1  3  4 23 24 21 11  5 20  6 19
[26] 18

# Vue que maintenant on a le temps pourquoi par regarder l'entropie avec RandomeForest 

In [3]:
#install.packages('randomForest')
library(randomForest)
library(tictoc)

Warning message:
"le package 'randomForest' a été compilé avec la version R 4.2.3"
randomForest 4.7-1.1

Type rfNews() to see new features/changes/bug fixes.


Attachement du package : 'randomForest'


L'objet suivant est masqué depuis 'package:dplyr':

    combine


Warning message:
"le package 'tictoc' a été compilé avec la version R 4.2.3"


In [4]:
data.rf<-read.csv("data_target_encoding.csv",stringsAsFactors = T)
test.rf<-read.csv("test_target_encoding.csv",stringsAsFactors = T)

In [5]:
colnames(data.rf)

[1] "geo_level_1_mean_damage"               
 [2] "geo_level_1_sd_damage"                 
 [3] "geo_level_2_mean_damage"               
 [4] "geo_level_2_sd_damage"                 
 [5] "geo_level_3_mean_damage"               
 [6] "geo_level_3_sd_damage"                 
 [7] "count_floors_pre_eq"                   
 [8] "age"                                   
 [9] "area_percentage"                       
[10] "height_percentage"                     
[11] "land_surface_condition"                
[12] "foundation_type"                       
[13] "roof_type"                             
[14] "ground_floor_type"                     
[15] "other_floor_type"                      
[16] "position"                              
[17] "plan_configuration"                    
[18] "has_superstructure_adobe_mud"          
[19] "has_superstructure_mud_mortar_stone"   
[20] "has_superstructure_stone_flag"         
[21] "has_superstructure_cement_mortar_stone"
[22] "has_superstructure_mud_mortar_brick"   
[23] "has_superstructure_cement_mortar_brick"
[24] "has_superstructure_timber"             
[25] "has_superstructure_bamboo"             
[26] "has_superstructure_rc_non_engineered"  
[27] "has_superstructure_rc_engineered"      
[28] "has_superstructure_other"              
[29] "legal_ownership_status"                
[30] "count_families"                        
[31] "has_secondary_use_agriculture"         
[32] "has_secondary_use_hotel"               
[33] "has_secondary_use_rental"              
[34] "has_secondary_use_institution"         
[35] "has_secondary_use_school"              
[36] "has_secondary_use_industry"            
[37] "has_secondary_use_health_post"         
[38] "has_secondary_use_gov_office"          
[39] "has_secondary_use_use_police"          
[40] "has_secondary_use_other"               
[41] "damage_grade"

In [7]:
tic()
set.seed(2)
n_trees <- 20
nfeat <- ncol(data.rf)-1 #I remove 1 to remove damage_grade
m_tries <- c(floor(0.5*sqrt(nfeat)))
nrows<-nrow(data.rf)
target_variable <- match('damage_grade', colnames(data.rf))

model <- randomForest(x=data.rf[,-c(target_variable)],
                    y=as.factor(data.rf[,c(target_variable)]),
                    ntree=n_trees,mtry=m_tries,keep.forest=TRUE,importance=TRUE)
pred.rf<-predict(model,test.rf[,-c(target_variable)])
toc()

113.78 sec elapsed


In [8]:
pred.rf <- as.numeric(pred.rf)

In [9]:
df.rf <- as.data.frame(pred.rf)
colnames(df.rf) <- c('damage_grade')

df.rf <- df.rf %>% group_by(damage_grade) %>% 
  summarise('H' =n (),
            .groups = 'drop')

for (i in 1:3){df.rf$H[i] <- df.rf$H[i]/86868}
df.rf

damage_grade,H
<dbl>,<dbl>
1,0.07124603
2,0.64775291
3,0.28100106


In [10]:
H <- - df.rf$H[1]*log(df.rf$H[1]) -df.rf$H[2]*log(df.rf$H[2]) -df.rf$H[3]*log(df.rf$H[3])
H

[1] 0.8261906

In [11]:
set.seed(2)
n_trees <- 20

pb1 <- txtProgressBar(min = 1, max = 40, style = 3)
    
for (i in 1:40){
    tic()
    data.rf.copy <- data.rf[,-i]
    test.rf.copy <- test.rf[,-i]
   

    nfeat <- ncol(data.rf.copy)-1 #I remove 1 to remove damage_grade
    m_tries <- c(floor(0.5*sqrt(nfeat)))
    nrows<-nrow(data.rf.copy)
    target_variable <- match('damage_grade', colnames(data.rf.copy))

    model <- randomForest(x=data.rf.copy[,-c(target_variable)],
                    y=as.factor(data.rf.copy[,c(target_variable)]),
                    ntree=n_trees,mtry=m_tries,keep.forest=TRUE,importance=TRUE)
    pred.rf.copy <- predict(model,test.rf.copy[,-c(target_variable)])
    
    ddf.rf <- as.data.frame(pred.rf.copy)
    colnames(ddf.rf) <- c('damage_grade')

    ddf.rf <- ddf.rf %>% group_by(damage_grade) %>% 
    summarise( 'A' = n(),
                .groups = 'drop')

    for (k in 1:3){ddf.rf$A[k] <- ddf.rf$A[k]/86868}

    df.rf <- merge(df.rf,ddf.rf,by=c('damage_grade','damage_grade'),all.x=T)
    names(df.rf)[i+2] = paste("H", i, sep = "")
    setTxtProgressBar(pb1, i)
    toc()
}
df.rf

  |                                                                      |   0%118.52 sec elapsed
  |==                                                                    |   3%108.61 sec elapsed
  |====                                                                  |   5%106.17 sec elapsed
  |=====                                                                 |   8%108.91 sec elapsed
  |=======                                                               |  10%107.89 sec elapsed
  |=========                                                             |  13%103.58 sec elapsed
  |===========                                                           |  15%107.95 sec elapsed
  |=============                                                         |  18%105.73 sec elapsed
  |==============                                                        |  21%105.14 sec elapsed
  |================                                                      |  23%105.75 sec elapsed
  |=================

damage_grade,H,H1,H2,H3,H4,H5,H6,H7,H8,⋯,H31,H32,H33,H34,H35,H36,H37,H38,H39,H40
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.07124603,0.07166045,0.0717065,0.06803426,0.07008334,0.06492609,0.07106184,0.0727886,0.07159138,⋯,0.07236267,0.07167196,0.07384768,0.07365198,0.07293825,0.07017544,0.07107335,0.07297279,0.07329511,0.07233964
2,0.64775291,0.64410370,0.6481098,0.65398075,0.65181655,0.65490169,0.64991712,0.6454736,0.64644058,⋯,0.64884653,0.65196620,0.65002072,0.64079983,0.64355114,0.65029700,0.64647511,0.64641755,0.64022425,0.64751117
3,0.28100106,0.28423585,0.2801837,0.27798499,0.27810011,0.28017222,0.27902104,0.2817378,0.28196804,⋯,0.27879081,0.27636184,0.27613160,0.28554819,0.28351061,0.27952756,0.28245154,0.28060966,0.28648064,0.28014919


In [12]:
entropie.rf <- 0:40
for (i in entropie.rf){ entropie.rf[i+1] <-  -df.rf[1,2+i]*log(df.rf[1,2+i]) -df.rf[2,2+i]*log(df.rf[2,2+i]) -df.rf[3,2+i]*log(df.rf[3,2+i])}
entropie.rf <- entropie.rf - entropie.rf[1] 
entropie.rf <- entropie.rf[2:41]
M.rf <- matrix(0,40,2)
M.rf[,1] <- entropie.rf
M.rf[,2] <- 1:40
M.rf <- as.data.frame(M.rf)
colnames(M.rf) <- c('entropie','indice')
M.rf <- M.rf[order(M.rf$entropie, decreasing = TRUE),]
M.rf

,entropie,indice
,<dbl>,<dbl>
34,0.0089941396,34
39,0.0089731730,39
20,0.0063593229,20
35,0.0057864491,35
21,0.0042155873,21
9,0.0041424489,9
11,0.0041244953,11
7,0.0039987612,7
1,0.0035862733,1


In [14]:
colnames(data.rf)[M.rf$indice]

[1] "has_secondary_use_institution"         
 [2] "has_secondary_use_use_police"          
 [3] "has_superstructure_stone_flag"         
 [4] "has_secondary_use_school"              
 [5] "has_superstructure_cement_mortar_stone"
 [6] "area_percentage"                       
 [7] "land_surface_condition"                
 [8] "count_floors_pre_eq"                   
 [9] "geo_level_1_mean_damage"               
[10] "legal_ownership_status"                
[11] "has_secondary_use_gov_office"          
[12] "roof_type"                             
[13] "foundation_type"                       
[14] "position"                              
[15] "has_superstructure_other"              
[16] "has_superstructure_rc_non_engineered"  
[17] "has_secondary_use_other"               
[18] "has_secondary_use_rental"              
[19] "age"                                   
[20] "has_superstructure_mud_mortar_stone"   
[21] "has_superstructure_mud_mortar_brick"   
[22] "has_secondary_use_health_post"         
[23] "has_secondary_use_agriculture"         
[24] "geo_level_1_sd_damage"                 
[25] "ground_floor_type"                     
[26] "has_superstructure_rc_engineered"      
[27] "has_superstructure_bamboo"             
[28] "has_superstructure_cement_mortar_brick"
[29] "height_percentage"                     
[30] "geo_level_3_sd_damage"                 
[31] "has_superstructure_adobe_mud"          
[32] "has_superstructure_timber"             
[33] "has_secondary_use_hotel"               
[34] "has_secondary_use_industry"            
[35] "count_families"                        
[36] "plan_configuration"                    
[37] "geo_level_2_sd_damage"                 
[38] "other_floor_type"                      
[39] "geo_level_2_mean_damage"               
[40] "geo_level_3_mean_damage"